# Join and preprocess multiple time series data

## Load data

In [4]:
import yfinance as yf
df = yf.download(tickers="MSFT", start='2018-01-01', end=None)

df

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT
Date,,,,,
2018-01-02,78.870369,79.200717,78.457438,79.035543,22483800
2018-01-03,79.237389,79.384213,78.888692,78.971275,26061400
...,...,...,...,...,...
2026-03-04,405.200012,411.029999,400.309998,401.269989,35750900
2026-03-05,408.554993,411.609985,404.399994,404.434998,13940531


In [5]:
df["Close"].plot()

In [6]:
import pandas as pd

path = "../../data/FRED/scripts/FRED data.csv"
df = pd.read_csv(filepath_or_buffer = path, index_col = "observation_date") 

df.plot()

## Parse dates to join time series

In [7]:
import pandas as pd

path1 = "../../data/FRED/scripts/FRED data.csv"
df1 = pd.read_csv(filepath_or_buffer = path1, index_col = 0, parse_dates=True) 

df1

,CORESTICKM159SFRBATL
observation_date,
1968-01-01,3.651861
1968-02-01,3.673819
...,...
2025-12-01,3.006319
2026-01-01,2.979751


In [8]:
df1.index

DatetimeIndex(['1968-01-01', '1968-02-01', '1968-03-01', '1968-04-01',
               '1968-05-01', '1968-06-01', '1968-07-01', '1968-08-01',
               '1968-09-01', '1968-10-01',
               ...
               '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01',
               '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01',
               '2025-12-01', '2026-01-01'],
              dtype='datetime64[ns]', name='observation_date', length=697, freq=None)

In [9]:
path2 = "../../data/FRED/scripts/MORTGAGE30US.csv"
df2 = pd.read_csv(filepath_or_buffer = path2, index_col = 0,parse_dates=True) 

df2

,MORTGAGE30US
observation_date,
1971-04-02,7.33
1971-04-09,7.31
...,...
2026-02-19,6.01
2026-02-26,5.98


In [10]:
df= pd.concat(objs=[df1, df2], axis=1)

df.plot()

## Inner join vs outer join

In [11]:
df= pd.concat(objs=[df1, df2], axis=1, join = "inner") #only data points that coincide are shown. 
df.tail(50).style


,CORESTICKM159SFRBATL,MORTGAGE30US
observation_date,,
1994-07-01 00:00:00,3.233880,8.570000
1995-09-01 00:00:00,3.351698,7.760000
1995-12-01 00:00:00,3.379829,7.330000
1996-03-01 00:00:00,3.134501,7.410000
1996-11-01 00:00:00,2.980390,7.780000
1997-08-01 00:00:00,2.653671,7.360000
1998-05-01 00:00:00,2.611332,7.220000
1999-10-01 00:00:00,2.049620,7.700000
2000-09-01 00:00:00,2.897082,7.960000


In [12]:
df.plot()

## Interpolate missing values

In [13]:
df= pd.concat(objs=[df1, df2], axis=1) 
df.tail(20).style



,CORESTICKM159SFRBATL,MORTGAGE30US
observation_date,,
2025-11-01 00:00:00,2.954815,nan
2025-11-06 00:00:00,nan,6.220000
2025-11-13 00:00:00,nan,6.240000
2025-11-20 00:00:00,nan,6.260000
2025-11-26 00:00:00,nan,6.230000
2025-12-01 00:00:00,3.006319,nan
2025-12-04 00:00:00,nan,6.190000
2025-12-11 00:00:00,nan,6.220000
2025-12-18 00:00:00,nan,6.210000


In [14]:
df = df.interpolate(method='linear')

df.tail(50).style



,CORESTICKM159SFRBATL,MORTGAGE30US
observation_date,,
2025-05-15 00:00:00,3.221797,6.810000
2025-05-22 00:00:00,3.248701,6.860000
2025-05-29 00:00:00,3.275605,6.890000
2025-06-01 00:00:00,3.302508,6.870000
2025-06-05 00:00:00,3.327421,6.850000
2025-06-12 00:00:00,3.352333,6.840000
2025-06-18 00:00:00,3.377245,6.810000
2025-06-26 00:00:00,3.402158,6.770000
2025-07-01 00:00:00,3.427070,6.720000


In [15]:
df.plot()

## Iterate for seamless load

In [16]:
paths = [
    "../../data/FRED/scripts/FRED data.csv",
    "../../data/FRED/scripts/MORTGAGE30US.csv",
    "../../data/FRED/scripts/FEDFUNDS.csv",
    "../../data/FRED/scripts/T10YIE.csv",
    "../../data/FRED/scripts/UNRATE.csv"
]

dfs = []
for path in paths:
    df = pd.read_csv(path, index_col = 0, parse_dates=True) 
    dfs.append(df)

df = pd.concat(objs=dfs, axis = 1).interpolate(method='linear')




In [17]:
df.tail(50).style

,CORESTICKM159SFRBATL,MORTGAGE30US,FEDFUNDS,T10YIE,UNRATE
observation_date,,,,,
2025-12-26 00:00:00,2.984371,6.168000,3.653913,2.230000,4.317391
2025-12-29 00:00:00,2.983216,6.162000,3.650435,2.220000,4.313043
2025-12-30 00:00:00,2.982061,6.156000,3.646957,2.240000,4.308696
2025-12-31 00:00:00,2.980906,6.150000,3.643478,2.250000,4.304348
2026-01-01 00:00:00,2.979751,6.151667,3.640000,2.250000,4.300000
2026-01-02 00:00:00,2.979751,6.153333,3.640000,2.250000,4.300000
2026-01-05 00:00:00,2.979751,6.155000,3.640000,2.260000,4.300000
2026-01-06 00:00:00,2.979751,6.156667,3.640000,2.270000,4.300000
2026-01-07 00:00:00,2.979751,6.158333,3.640000,2.270000,4.300000


In [18]:
df.plot()

## Export data

In [19]:
df

,CORESTICKM159SFRBATL,MORTGAGE30US,FEDFUNDS,T10YIE,UNRATE
observation_date,,,,,
1948-01-01,NaN,NaN,NaN,NaN,3.4
1948-02-01,NaN,NaN,NaN,NaN,3.8
...,...,...,...,...,...
2026-03-03,2.979751,5.98,3.64,2.29,4.3
2026-03-04,2.979751,5.98,3.64,2.29,4.3


In [20]:
df.index.name = "Date"

df

,CORESTICKM159SFRBATL,MORTGAGE30US,FEDFUNDS,T10YIE,UNRATE
Date,,,,,
1948-01-01,NaN,NaN,NaN,NaN,3.4
1948-02-01,NaN,NaN,NaN,NaN,3.8
...,...,...,...,...,...
2026-03-03,2.979751,5.98,3.64,2.29,4.3
2026-03-04,2.979751,5.98,3.64,2.29,4.3


In [21]:
df= df.rename(columns ={
    "CORESTICKM159SFRBATL": "CPI",
    "MORTGAGE30US": "30 Year Mortgage Rate",
    "FEDFUNDS": "Federal Funds Rate",
    "T10YIE": "10 Year Treasury Inflation Expectation",
    "UNRATE": "Unemployment Rate"
})

In [22]:
df.plot()

In [24]:
import pandas as pd
df.to_excel('../../data/FRED/scripts/FRED_Data/FRED_joined.xlsx')

ModuleNotFoundError: No module named 'openpyxl'

In [ ]:
df.to_csv('../../data/FRED/scripts/FRED_Data/FRED_joined.csv') #won't be saved exactly as created in notebook

In [27]:
df.to_parquet("../../data/FRED/scripts/FRED_Data/FRED_joined.parquet")
pd.read_parquet("../../data/FRED/scripts/FRED_Data/FRED_joined.parquet").index 

DatetimeIndex(['1948-01-01', '1948-02-01', '1948-03-01', '1948-04-01',
               '1948-05-01', '1948-06-01', '1948-07-01', '1948-08-01',
               '1948-09-01', '1948-10-01',
               ...
               '2026-02-19', '2026-02-20', '2026-02-23', '2026-02-24',
               '2026-02-25', '2026-02-26', '2026-02-27', '2026-03-02',
               '2026-03-03', '2026-03-04'],
              dtype='datetime64[ns]', name='Date', length=4727, freq=None)